In [ ]:
from pathlib import Path
import sys
sys.path.append(str(Path().resolve().parent / 'src'))

In [ ]:
from typing import Any, List

from data_handlers.echr_data_handler import EchrDataHandler
from utils.evaluation_utils import EvaluationUtils
from utils.project_utils import ProjectUtils

import pandas as pd

In [ ]:
project_root: Path = ProjectUtils.get_project_root(); project_root

In [ ]:
echr_data_handler: EchrDataHandler = EchrDataHandler(project_root)
echr_raw_file_names: List[str] = echr_data_handler.get_available_raw_files()

In [ ]:
k = 1
replacement_strategy="semantic_label_mask"

model_name = 'xlm-roberta-large'
data_dir_path = Path(f'/home/ssaha/model-checkpoints/echr/tc/{model_name}/additional-embeddings-none/sample-size-8435/data-fold-{k}')
model_file_path = data_dir_path / 'learning-rate-5e-7' / 'max-epochs-25' / 'mini-batch-size-2' / 'best-model.pt'

input_df = echr_data_handler.get_train_dev_test_datasetdict(k=k)["test"].to_pandas()
pe_df = echr_data_handler.get_private_entities_df(echr_raw_file_names[0])
id_column="itemid"
text_column="text"
class_column="binary_judgement"
pe_column="text_pe_ontonotes5_ner-english-ontonotes-large"
zero_entity_retain_text=True

result = EvaluationUtils.redact_and_evaluate_for_text_classifier(input_df=input_df,
                                                                 pe_df=pe_df,
                                                                 id_column=id_column,
                                                                 text_column=text_column,
                                                                 class_column=class_column,
                                                                 pe_column=pe_column,
                                                                 replacement_strategy=replacement_strategy,
                                                                 zero_entity_retain_text=zero_entity_retain_text,
                                                                 data_dir_path=data_dir_path,
                                                                 model_file_path=model_file_path)
print(result.detailed_results)